<div dir="rtl">
<h1>آینده را عوض کنید، گذشته را بسنجید</h1>
<p>درس 65 از 76 · یک قطعه را حذف کنیم و یک ادعا را بیازماییم · <code dir="ltr">58-ablation</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-04/58-ablation.html">📖 بازگشت به همین درس</a></p>
<p>یک آزمون علّیت بنویسید که تنها یک کلید محاسبه را تغییر می‌دهد.</p><p>پیش‌نیاز: Causal Mask و مدل در حالت eval.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>با ثابت‌ماندن سه Token نخست، تغییر دو Token آخر باید کدام Logits را ثابت نگه دارد؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import copy
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
model = MiniGPT(ModelConfig(12,8,16,2,2,0.0)).eval()
a,b = torch.tensor([[1,2,3,4,5]]),torch.tensor([[1,2,3,9,10]])
with torch.no_grad():
    trace = {}
    model(a,trace=trace)
print('future attention mass:',trace['layers'][0]['attention']['weights'].triu(1).sum().item())

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>past_difference(Model, a, b, prefix, causal) در eval/no_grad، بیشترین قدرمطلق اختلاف Logits نخستین prefix موقعیت را برگرداند. مقدار causal را به هر دو forward بدهید و حالت قبلی مدل را برگردانید.</p>
</div>

In [ ]:
def past_difference(model, a, b, prefix, causal):
    # TODO: ورودی‌ها فقط در آینده متفاوت‌اند
    return None

In [ ]:
def test_exercise():
    model.train()
    result = past_difference(model,a,b,3,True)
    if result is None:
        return False
    assert result<1e-7
    assert model.training
    assert past_difference(model,a,b,3,False)>1e-6
    assert past_difference(model,a,a,3,False)==0
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: past_difference')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط use_positions را عوض کنید؛ در هر حالت دو جایگشت با Token آخر یکسان را مقایسه کنید. صفرشدن تفاوت را پیش‌فرض نگیرید.</p>
</div>

In [ ]:
model.eval()
with torch.no_grad():
    for enabled in (False,True):
        left = model(torch.tensor([[1,2,3,4]]),use_positions=enabled)[0][:,-1]
        right = model(torch.tensor([[3,2,1,4]]),use_positions=enabled)[0][:,-1]
        print('positions:',enabled,'last difference:',(left-right).abs().max().item())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>دو مدل تصادفی مستقل، مقایسهٔ کنترل‌شده نیستند. paired_models(Model) دو deepcopy مستقل از یک وضعیت یکسان برگرداند؛ تغییر یک نسخه نباید روی دیگری یا مدل پایه اثر بگذارد.</p>
</div>

In [ ]:
left = MiniGPT(model.config)
right = MiniGPT(model.config)
print('independent initial weights equal:',torch.equal(left.token_embedding.weight,right.token_embedding.weight))

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def paired_models(model):
    # TODO: وزن آغازین برابر، حافظهٔ مستقل
    return None

In [ ]:
def test_repair():
    result = paired_models(model)
    if result is None:
        return False
    left,right = result
    assert left is not right and left is not model and right is not model
    assert all(torch.equal(left.state_dict()[name],value) for name,value in right.state_dict().items())
    with torch.no_grad():
        left.token_embedding.weight.add_(1)
    assert torch.equal(right.token_embedding.weight,model.token_embedding.weight)
    assert not torch.equal(left.token_embedding.weight,right.token_embedding.weight)
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: paired_models')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>causal و use_positions کلیدهای آزمایشی forward واقعی‌اند؛ train معمولی هر دو را فعال می‌گذارد. تغییر خروجی مدل تصادفی، برتری کیفیت پس از آموزش را ثابت نمی‌کند.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>آزمایش علّیت کدام ادعای ساختاری را می‌سنجد که یک متن تولیدی روان نمی‌تواند ثابت کند؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-04/58-ablation.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/58-ablation.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>